Let's do a text frequency analyzer

In [16]:
from collections import Counter
import re

def text_frequency(text):
  text = text.lower()
  tokens = re.findall(r'\b[a-z]+\b',text)
  return Counter(tokens)

In order to reduce noise and focus on meaningful terms we will remove stop words using the nltk (The Natural Language Toolkit) used for NLP

In [17]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
stop_word = set(stopwords.words('english'))
print(list(stop_word))

['had', 'each', 'not', "you'd", 'below', 'didn', 'myself', "they're", 'were', 're', 'what', 'your', 'any', 'by', 'd', "haven't", 'for', 'during', 'herself', 'it', "weren't", 'he', 'shouldn', 'which', "it'd", 'with', 'most', "you're", "couldn't", 'but', 'ourselves', 'its', 'a', 'who', 'ma', 'they', 'before', 'just', 'she', "hasn't", 'being', 'these', 'why', "isn't", 'does', 'or', 'an', "needn't", 'yourself', 'been', 'o', "mightn't", 's', 'under', 'so', 'above', 'between', 'all', 'couldn', 'ain', 'doesn', 'on', 'from', 't', 'the', 'isn', 'am', 'other', 'wouldn', 'you', "shouldn't", 'to', 'after', 'than', "he'll", 'then', 'do', 'again', "we'd", "he'd", 'haven', "he's", 'those', "hadn't", "i've", 'there', 'until', 'don', "shan't", 'few', 'off', 'mightn', "they'd", 'yours', 'against', 'and', 'be', 'through', "won't", "didn't", 'over', 'himself', 'if', 'hasn', 'no', 'needn', "you'll", 'into', 'both', 'doing', "she'd", 'can', 'themselves', 'nor', "don't", 'll', 'of', "i'm", 'did', 'further', 

In [19]:
def text_frequency_filtered(text):
  text = text.lower()
  tokens = re.findall(r'\b[a-z]+\b',text)
  filtered = [word for word in tokens if word not in stop_words]
  return Counter(filtered)


For more meaningful insights, we'll return the top_N frequent words

In [22]:
from collections import Counter
from nltk.corpus import stopwords
import re
stop_words = set(stopwords.words('english'))
def text_frequency_rank(text, N):
  text = text.lower()
  tokens = re.findall(r'\b[a-z]+\b',text)
  filtered = [word for word in tokens if word not in stop_words]
  count = Counter(filtered)
  sorted_items = sorted(count.items(),key = lambda x : x[1],reverse = True) # sort the items(key, value) by value
  top_N = sorted_items[:N]
  return top_N


Apply this function to a real dataset column (CSV or scraped text) Then extend it to: remove stopwords compute top-N words per document compare two texts (similarity idea) That is where this becomes actual data science instead of coding drills.

In [23]:
txt = "Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum."
print(text_frequency(txt,10))

[('lorem', 4), ('ipsum', 4), ('dummy', 2), ('text', 2), ('typesetting', 2), ('industry', 2), ('type', 2), ('simply', 1), ('printing', 1), ('standard', 1)]


Our pipeline is now correct up to feature extraction and ranking. now we can turn text into comparable vectors.

1- Jaccard Similarity : Compare overlap between two sets of words
J(A,B)=∣A∩B∣/∣A∪B∣


In [28]:
def text_preprocess(text):
  text = text.lower()
  tokens = re.findall(r'\b[a-z]+\b',text)
  filtered = [word for word in tokens if word not in stop_words]
  return set(filtered)

def jaccard_similarity(t1, t2):
  A = text_preprocess(t1)
  B = text_preprocess(t2)
  Intersection = A & B
  Union = A | B
  return len(Intersection) / len(Union)

In [31]:
t1 = "I am into data science and machine learning"
t2 = "machine learning is part of data science"
t3 = "I need to learn how to use the sewing machine"
print(jaccard_similarity(t1,t2))
print(jaccard_similarity(t1,t3))

0.8
0.125


2- Cosine similarity : vectorize text into a Bag of Words
Cos(𝐴,𝐵)=𝐴⋅𝐵/(‖𝐴‖‖𝐵‖)

In [35]:
import numpy as np

def vectorize (text, vocab):
  tokens = text_preprocess(text)
  count = Counter(tokens)
  return  np.array([count.get(word,0) for word in vocab]) #create an array of words in vocab and their count in text


def cos_similarity(A, B):
  dot = np.dot(A,B)
  norm_A = np.linalg.norm(A)
  norm_B = np.linalg.norm(B)
  if norm_A == 0 or norm_B == 0 :
    return 0

  return dot/(norm_A * norm_B)

t1 = "I am into data science and machine learning"
t2 = "machine learning is part of data science"
vocab1 = list(text_preprocess(t1) | text_preprocess(t2))

v1 = vectorize(t1, vocab)
v2 = vectorize(t2, vocab)

print(cos_similarity(v1, v2))



0.8944271909999159
